# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [4]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [5]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from dotenv import load_dotenv


In [6]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [7]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

from pydantic import BaseModel, Field
from typing import List, Any, Annotated

class GameResult(BaseModel):
    """Represents a single Game query result"""
    platform: Annotated[str, Field(description="the platform of the game (Game Boy, Playstation 5, Xbox 360...)")]
    name: Annotated[str, Field(description="the name of the game")]
    year_of_release: Annotated[int, Field(description="when the game was released for that platform")]
    description: Annotated[str, Field(description="Additional details about the game")]

@tool
def retrieve_game_tool(query:str) -> List[GameResult]:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry. 
    """
    chroma_client = chromadb.PersistentClient(path="chromadb")
    collection = chroma_client.get_collection("udaplay")
    results:Run = collection.query(query)
    return results

#### Evaluate Retrieval Tool

In [8]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

from lib.evaluation import TestCase, AgentEvaluator, EvaluationResult

evaluator = AgentEvaluator()

class EvaluationReport(BaseModel):
    """Represents the evaluation of retrieval"""
    usefulness: Annotated[bool, Field(description="whether the documents are useful to answer the question")]
    description: Annotated[str, Field(description="description about the evaluation result")]

@tool
def evaluate_retrieval_tool(question:str, retrieved_docs:List[str]) -> EvaluationReport:
    """ 
    Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question. 
    """
    judge_prompt = f"""
    Your task is to evaluate if the documents {retrieved_docs} are enough to respond the query {question}.
    Give a detailed explanation, so it's possible to take an action to accept it or not.
    """
    llm_judge = LLM(model="gpt-4o-mini")
    ai_message = llm_judge.invoke(input=judge_prompt, response_format=EvaluationReport)
    parser = JsonOutputParser()
    return parser.parse(ai_message)


#### Game Web Search Tool

In [18]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

import requests


@tool
def game_web_search_tool(question:str) -> str:
    """
    Semantic search: Finds most results in the vector DB
    """
    API_KEY = os.getenv("TAVILY_API_KEY")
    BASE_URL = "https://app.tavily.com"
    response = requests.get(BASE_URL)
    response.raise_for_status()
    return response

    

### Agent

In [19]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

tools = [retrieve_game_tool, evaluate_retrieval_tool, game_web_search_tool]

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
        "You are an assistant that can help with \n"
        "1. retrieving games from a database \n"
        "2. evaluating your own responses \n"
        "3. performing web searches on games \n"
        "Use the available tools to help answer questions about these topics \n"
        "Maintain context across conversations within the same session" 
    ),
    tools=tools
)

In [20]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
from lib.messages import BaseMessage

def print_messages(messages: List[BaseMessage]):
    for m in messages:
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

session_id = "session_1"

run1 = agent.invoke(
    query="When were Pokémon Gold and Silver released?",
    session_id=session_id
)

print("\nMessages from run 1:")
messages = run1.get_final_state()["messages"]
print_messages(messages)

run2 = agent.invoke(
    query="Which one was the first 3D platformer Mario game",
    session_id=session_id
)

print("\nMessages from run 2:")
messages = run2.get_final_state()["messages"]
print_messages(messages)

run3 = agent.invoke(
    query="Was Mortal Kombat X realeased for Playstation 5?",
    session_id=session_id
)

print("\nMessages from run 3:")
messages = run3.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 1:
 -> (role = system, content = You are an assistant that can help with 
1. retrieving games from a database 
2. evaluating your own responses 
3. performing web searches on games 
Use the available tools to help answer questions about these topics 
Maintain context across conversations within the same session, tool_calls = None)
 -> (role = user, content = When were Pokémon Gold and Silver released?, tool_calls = None)
 -> (role = assistant, content = Pokémon Gold and Silver were released in Japan on November 21, 1999. They were later released in North America on October 16, 2000, and in Europe on April 6, 2001., tool_calls = None)
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Me

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes